# Week 5.1 — Steepest Descent vs. Conjugate Gradient (SPD System)
Compares two gradient-based methods for solving $Ax=b$ with SPD $A$: plain steepest descent, which can zig-zag badly on ill-conditioned problems, against Conjugate Gradient, which is guaranteed to converge in at most $n$ steps in exact arithmetic and is far less sensitive to conditioning in practice.

In [1]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

## Helper functions
`steepest_descent` moves along the residual direction with the exact line-search step $\alpha=\frac{r^Tr}{r^TAr}$ that minimizes the quadratic along that direction. `cg_spd` implements the standard Conjugate Gradient recurrence, which instead builds a sequence of $A$-conjugate search directions $p_k$ so that each step makes optimal progress without undoing previous steps.

In [2]:
def steepest_descent(A, b, x0, tol, maxit):
    x = x0.copy()
    r = b - A @ x
    reshist = [np.linalg.norm(r)]
    k = 0
    while reshist[-1] > tol and k < maxit:
        Ar = A @ r
        alpha = (r.T @ r) / (r.T @ Ar)  # exact line-search step
        x = x + alpha * r
        r = r - alpha * Ar
        reshist.append(np.linalg.norm(r))
        k += 1
    return x, np.array(reshist)

def cg_spd(A, b, x0, tol, maxit):
    x = x0.copy()
    r = b - A @ x
    p = r.copy()
    rho = r.T @ r
    reshist = [np.sqrt(rho)]
    k = 0
    while reshist[-1] > tol and k < maxit:
        Ap = A @ p
        alpha = rho / (p.T @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rho_new = r.T @ r
        beta = rho_new / rho
        p = r + beta * p
        rho = rho_new
        reshist.append(np.sqrt(rho))
        k += 1
    return x, np.array(reshist)

## Problem setup: Steepest Descent vs. CG on SPD (1D Poisson)
Uses the same tridiagonal 1D Poisson matrix as in Week 3/7 (SPD, moderately ill-conditioned for $n=400$) with a random right-hand side, so the two methods can be compared on a problem where conditioning actually matters.

In [3]:
n = 400
e = np.ones(n)
A = sp.diags([-e, 2 * e, -e], [-1, 0, 1], shape=(n, n), format='csr')  # 1D Poisson (Dirichlet)
b = np.random.randn(n)                            # generic RHS
x0 = np.zeros(n)

tol = 1e-8
maxit = 2000

## Run solvers
Runs both methods from the same zero initial guess to the same tolerance.

In [4]:
x_sd, res_sd = steepest_descent(A, b, x0, tol, maxit)
x_cg, res_cg = cg_spd(A, b, x0, tol, maxit)

## Compare residual histories
Plots both residual histories on a log scale. Steepest descent's convergence rate depends on $\left(\frac{\kappa-1}{\kappa+1}\right)^2$ where $\kappa=\text{cond}(A)$, so it slows sharply as conditioning worsens, while CG's rate depends on $\frac{\sqrt{\kappa}-1}{\sqrt{\kappa}+1}$ — the square root of the same condition number — which is why it needs far fewer iterations here.

In [5]:
plt.figure()
plt.semilogy(res_sd, 'o-', label='Steepest Descent')
plt.semilogy(res_cg, 'x-', label='CG')
plt.grid(True)
plt.xlabel('Iteration')
plt.ylabel('||r_k||_2')
plt.title('Steepest Descent vs. Conjugate Gradient on SPD system')
plt.legend()
plt.show()

print(f"Steepest Descent iterations: {len(res_sd)-1}")
print(f"Conjugate Gradient iterations: {len(res_cg)-1}")

Steepest Descent iterations: 2000
Conjugate Gradient iterations: 400


/var/folders/k6/1w07pxzj0mx129drg82_k3_w0000gp/T/ipykernel_73665/1235910290.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
